# 01 — Explore & quality-check openFDA drug/event (bronze)

Step 1 of the pipeline: **look** at the raw data and run the 5 data-quality checks
(Completeness, Accuracy, Consistency, Timeliness, Uniqueness) — the week-12 pattern.
This only reads the data; it changes nothing.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('openfda_explore')
         .config('spark.driver.memory', '4g')
         .getOrCreate())
spark

In [2]:
# Read all days. For a FAST first run, point at one day, e.g. receivedate=20240102
DATA_GLOB = '/home/jovyan/data/bronze/drug_event/receivedate=*/*.json'
df = spark.read.json(DATA_GLOB)
total = df.count()
print('Total rows:', total)

Total rows: 2687675


## 1. Structure — what fields, and how nested?

In [3]:
df.printSchema()

root
 |-- authoritynumb: string (nullable = true)
 |-- companynumb: string (nullable = true)
 |-- duplicate: string (nullable = true)
 |-- fulfillexpeditecriteria: string (nullable = true)
 |-- occurcountry: string (nullable = true)
 |-- patient: struct (nullable = true)
 |    |-- drug: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- actiondrug: string (nullable = true)
 |    |    |    |-- activesubstance: struct (nullable = true)
 |    |    |    |    |-- activesubstancename: string (nullable = true)
 |    |    |    |-- drugadditional: string (nullable = true)
 |    |    |    |-- drugadministrationroute: string (nullable = true)
 |    |    |    |-- drugauthorizationnumb: string (nullable = true)
 |    |    |    |-- drugbatchnumb: string (nullable = true)
 |    |    |    |-- drugcharacterization: string (nullable = true)
 |    |    |    |-- drugcumulativedosagenumb: string (nullable = true)
 |    |    |    |-- drugcumulativedosageunit: st

## 2. Uniqueness — is each `safetyreportid` one row?  →  feeds #4 dedup

In [ ]:
distinct_ids = df.select('safetyreportid').distinct().count()
print('rows:', total)
print('distinct safetyreportid:', distinct_ids)
print('duplicate rows:', total - distinct_ids)

dups = df.groupBy('safetyreportid').count().filter('count > 1')
print('safetyreportids appearing more than once:', dups.count())
dups.orderBy(F.desc('count')).show(5)

## 3. Completeness — how often are key fields missing?  →  feeds #6

In [ ]:
for c in ['safetyreportid', 'safetyreportversion', 'receivedate', 'serious', 'patient']:
    n = df.filter(F.col(c).isNull()).count()
    print(f'{c:20s} {100*n/total:6.2f}% null')

## 4. Consistency — do coded fields hold only valid values?  →  feeds #6
`serious` should be 1 or 2; `patientsex` should be 0, 1 or 2.

In [ ]:
df.groupBy('serious').count().orderBy('serious').show()
df.select(F.col('patient.patientsex').alias('patientsex')).groupBy('patientsex').count().orderBy('patientsex').show()

## 5. Timeliness — oldest and newest `receivedate`

In [ ]:
df.select(F.min('receivedate').alias('oldest'), F.max('receivedate').alias('newest')).show()

## 6. Drug-name messiness — top `medicinalproduct` values  →  feeds #5 normalisation
One report has many drugs, so we `explode` `patient.drug` into one row per drug first.

In [ ]:
drugs = df.select(F.explode('patient.drug').alias('d'))
print('total drug rows:', drugs.count())
drugs.select('d.medicinalproduct').groupBy('medicinalproduct').count().orderBy(F.desc('count')).show(20, truncate=False)

## 7. Fan-out — how many drugs & reactions per report?

In [ ]:
df.select(F.size('patient.drug').alias('n_drugs'),
          F.size('patient.reaction').alias('n_reactions')).describe().show()

## Notes — write what you found here

- Duplicates: ...
- Most-null fields: ...
- Drug-name mess: ...
- Drugs / reactions per report: ...

These findings define the cleaning work: **#4 dedup**, **#5 normalisation**, **#6 validation**.